In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

train        = pd.read_csv('../data/train.csv', parse_dates=['date'])
test         = pd.read_csv('../data/test.csv',  parse_dates=['date'])
stores       = pd.read_csv('../data/stores.csv')
oil          = pd.read_csv('../data/oil.csv',   parse_dates=['date'])
holidays     = pd.read_csv('../data/holidays_events.csv', parse_dates=['date'])
transactions = pd.read_csv('../data/transactions.csv',    parse_dates=['date'])

# Oil eksiklerini doldur
oil['dcoilwtico'] = oil['dcoilwtico'].ffill().bfill()

# Tüm dosyaları birleştir
ulusal_tatiller = holidays[holidays['locale'] == 'National'][['date','type']].copy()
ulusal_tatiller = ulusal_tatiller.rename(columns={'type': 'tatil_tipi'})
ulusal_tatiller = ulusal_tatiller.drop_duplicates(subset='date')

df = train.merge(stores,       on='store_nbr',         how='left')
df = df.merge(oil,             on='date',               how='left')
df = df.merge(transactions,    on=['date','store_nbr'], how='left')
df = df.merge(ulusal_tatiller, on='date',               how='left')

df['dcoilwtico']  = df['dcoilwtico'].ffill().bfill()
df['transactions'] = df['transactions'].fillna(0)
df['tatil_tipi']  = df['tatil_tipi'].fillna('Normal')

print(f"EDA'da yaptıklarımı yeni klasör için hazır hale getirdik (veri hazır): {df.shape}")
df.columns

EDA'da yaptıklarımı yeni klasör için hazır hale getirdik (veri hazır): (3000888, 13)


Index(['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'city',
       'state', 'type', 'cluster', 'dcoilwtico', 'transactions', 'tatil_tipi'],
      dtype='str')

In [ ]:
#Tarih değişkenleirini model diline çevir
df['yil']          = df['date'].dt.year
df['ay']           = df['date'].dt.month
df['gun']          = df['date'].dt.day
df['haftanin_gunu'] = df['date'].dt.dayofweek

df['hafta_sonu']   = (df['haftanin_gunu'] >= 5).astype(int) # cmts[5],pazar[6] -> 1, diğerleri = 0 (astype(int) → True/False'u 1/0'a çevirir)

df['ayin_haftasi'] = df['date'].dt.isocalendar().week.astype(int) #(yılın kaçıncı haftası — mevsimsellik için)

print("Tarih değişkenleri eklendi:")
print(df[['date','yil','ay','gun','haftanin_gunu','hafta_sonu']].head())

Tarih değişkenleri eklendi:
        date   yil  ay  gun  haftanin_gunu  hafta_sonu
0 2013-01-01  2013   1    1              1           0
1 2013-01-01  2013   1    1              1           0
2 2013-01-01  2013   1    1              1           0
3 2013-01-01  2013   1    1              1           0
4 2013-01-01  2013   1    1              1           0


In [18]:
# Lag değişkenleri (7gün - 14gün.. önce ne kadar satıldı) (zaman serisi modellerinin temel dayanağı)
df = df.sort_values(['store_nbr','family','date'])
df['lag_7'] = df.groupby(['store_nbr','family'])['sales'].shift(7)
df['lag_14'] = df.groupby(['store_nbr','family'])['sales'].shift(14)
df['lag_28'] = df.groupby(['store_nbr','family'])['sales'].shift(28)
print(f"Lag Değişkenleri: \n {df[['date','store_nbr','family','sales','lag_7','lag_14','lag_28']].head(20)}")

Lag Değişkenleri: 
             date  store_nbr      family  sales  lag_7  lag_14  lag_28
0     2013-01-01          1  AUTOMOTIVE    0.0    NaN     NaN     NaN
1782  2013-01-02          1  AUTOMOTIVE    2.0    NaN     NaN     NaN
3564  2013-01-03          1  AUTOMOTIVE    3.0    NaN     NaN     NaN
5346  2013-01-04          1  AUTOMOTIVE    3.0    NaN     NaN     NaN
7128  2013-01-05          1  AUTOMOTIVE    5.0    NaN     NaN     NaN
8910  2013-01-06          1  AUTOMOTIVE    2.0    NaN     NaN     NaN
10692 2013-01-07          1  AUTOMOTIVE    0.0    NaN     NaN     NaN
12474 2013-01-08          1  AUTOMOTIVE    2.0    0.0     NaN     NaN
14256 2013-01-09          1  AUTOMOTIVE    2.0    2.0     NaN     NaN
16038 2013-01-10          1  AUTOMOTIVE    2.0    3.0     NaN     NaN
17820 2013-01-11          1  AUTOMOTIVE    3.0    3.0     NaN     NaN
19602 2013-01-12          1  AUTOMOTIVE    2.0    5.0     NaN     NaN
21384 2013-01-13          1  AUTOMOTIVE    2.0    2.0     NaN     NaN


In [ ]:
#Rolling ortalama (Lag tek bir günü yakalar - Rolling ortalama genel trendi yakalar(daha kararlı)) son 7 veya 14 günün satışı
#shift(1)-> sızdırma(data leakage) olmaması için bugünün satışını dahil etme 
#.transform(lambda x: ...) --> Her gruba bir fonksiyon uygula lambda = tek satırlık küçük fonksiyon demek x: o grubun sales serisi
df['rolling_7']  = df.groupby(['store_nbr','family'])['sales'].transform(lambda x: x.shift(1).rolling(7).mean())

df['rolling_14'] = df.groupby(['store_nbr','family'])['sales'].transform(lambda x: x.shift(1).rolling(14).mean())

print(f"Rolling ortalamalar: \n {df[['date','store_nbr','family','sales','rolling_7','rolling_14']]}")

Rolling ortalamalar: 
               date  store_nbr      family  sales  rolling_7  rolling_14
0       2013-01-01          1  AUTOMOTIVE    0.0        NaN         NaN
1782    2013-01-02          1  AUTOMOTIVE    2.0        NaN         NaN
3564    2013-01-03          1  AUTOMOTIVE    3.0        NaN         NaN
5346    2013-01-04          1  AUTOMOTIVE    3.0        NaN         NaN
7128    2013-01-05          1  AUTOMOTIVE    5.0        NaN         NaN
...            ...        ...         ...    ...        ...         ...
2993627 2017-08-11         54     SEAFOOD    0.0   3.000000    3.428571
2995409 2017-08-12         54     SEAFOOD    1.0   3.000000    3.142857
2997191 2017-08-13         54     SEAFOOD    2.0   2.714286    2.928571
2998973 2017-08-14         54     SEAFOOD    0.0   3.000000    2.785714
3000755 2017-08-15         54     SEAFOOD    3.0   3.000000    2.500000

[3000888 rows x 6 columns]


In [ ]:
#Encoding - Model metin okuyamaz bunları sayıya çevirmemiz gerekiyor (her kategoriye sayı ata)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

df['family_enc']     = le.fit_transform(df['family'])
df['city_enc']       = le.fit_transform(df['city'])
df['state_enc']      = le.fit_transform(df['state'])
df['type_enc']       = le.fit_transform(df['type'])
df['tatil_enc']      = le.fit_transform(df['tatil_tipi'])
df['promosyon_var']  = (df['onpromotion'] > 0).astype(int)

print(f"Encoding tamam: \n {df[['family','family_enc','city','city_enc','type','type_enc',]].head()}") #sadece encoding doğru mu oldu mu kontrol


Encoding tamamlandı:
Encoding tamam: 
           family  family_enc   city  city_enc type  type_enc
0     AUTOMOTIVE           0  Quito        18    D         3
1782  AUTOMOTIVE           0  Quito        18    D         3
3564  AUTOMOTIVE           0  Quito        18    D         3
5346  AUTOMOTIVE           0  Quito        18    D         3
7128  AUTOMOTIVE           0  Quito        18    D         3


In [29]:
print(f"Final shape: {df.shape} \n Sütunlar: {df.columns.tolist()}")
eksik = df.isnull().sum()
print(f"Eksik değerler: \n {eksik[eksik > 0]}") #lag ve rollingler'de eksikler var çünkü ilk 28 biçin geçmiş yok 
df.to_csv('../data/features.csv', index=False)
print("Feature engineering tamamlandı, features.csv kaydedildi!")



Final shape: (3000888, 30) 
 Sütunlar: ['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'city', 'state', 'type', 'cluster', 'dcoilwtico', 'transactions', 'tatil_tipi', 'yil', 'ay', 'gun', 'haftanin_gunu', 'hafta_sonu', 'ayin_haftasi', 'lag_7', 'lag_14', 'lag_28', 'rolling_7', 'rolling_14', 'family_enc', 'city_enc', 'state_enc', 'type_enc', 'tatil_enc', 'promosyon_var']
Eksik değerler: 
 lag_7         12474
lag_14        24948
lag_28        49896
rolling_7     12474
rolling_14    24948
dtype: int64
Feature engineering tamamlandı, features.csv kaydedildi!
